# Deteksi Fault 4 Sensor Kelembaban Tanah: EDM-Fuzzy vs JSD-Fuzzy + ANN-LM

**Notebook sidang interaktif.** Semua angka dibaca dari hasil penelitian yang sudah
diekspor; tidak ada model yang dilatih ulang di sini, jadi tiap sel selesai dalam
hitungan detik.

Cara pakai: **Run All** sekali, lalu ubah pilihan di panel kontrol (hari, sensor,
jenis fault, intensitas). Grafik dan tebakan model ikut berubah.

In [ ]:
# === Setup 1/3: pustaka angka ===
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd

In [ ]:
# === Setup 2/3: grafik dan widget ===
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, HTML, Markdown

In [ ]:
# === Setup 3/3: lokasi file, mode presentasi ===

# Lokal: repo penelitian. Kaggle: file dataset `sidang-bahan` (datar) disusun ulang
# ke struktur folder repo di /kaggle/working/bahan memakai symlink.
BERKAS = {"cache_interaktif.npz": "sidang/assets", "model_sidang.npz": "sidang/assets",
          "11_rq5_f1_berpasangan.csv": "hasil11/exports", "11_sensor_ringkas.csv": "hasil11/exports",
          "statistik_sel.csv": "sidang/qa/hasil_uji", "statistik_lipatan.csv": "sidang/qa/hasil_uji",
          "efek_purge_permutasi.csv": "sidang/qa/hasil_uji"}
ROOT = next((p for p in [Path(".."), Path(".")]
             if (p / "sidang" / "assets" / "cache_interaktif.npz").exists()), None)
if ROOT is None:
    import glob
    ROOT = Path("/kaggle/working/bahan")
    for nama, folder in BERKAS.items():
        asal = glob.glob(f"/kaggle/input/**/{nama}", recursive=True)
        if asal:
            (ROOT / folder).mkdir(parents=True, exist_ok=True)
            tujuan = ROOT / folder / nama
            if not tujuan.exists():
                tujuan.symlink_to(asal[0])
C = np.load(ROOT / "sidang" / "assets" / "cache_interaktif.npz")
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

_gaya = w.Output()
def _mode(presentasi):
    with _gaya:
        _gaya.clear_output()
        if presentasi:
            display(HTML("<style>.catatan-internal{display:none!important}</style>"))
tombol_mode = w.ToggleButton(value=False, description="Mode presentasi", icon="eye-slash",
                             tooltip="Sembunyikan semua catatan internal")
tombol_mode.observe(lambda ch: _mode(ch["new"]), "value")
display(w.HBox([tombol_mode, _gaya]))
print("Folder kerja:", ROOT.resolve())

## 1. Masalahnya

Sensor kelembaban tanah di sistem irigasi bisa rusak diam-diam: nilainya bergeser,
melonjak, merayap naik, atau macet. Sistem yang percaya pada sensor rusak akan
menyiram terlalu banyak atau terlalu sedikit.

**Pertanyaan penelitian:** kalau sinyal 4 sensor dipotong per jendela waktu dan
tiap sensor diringkas dengan **entropi multiskala**, apakah mengganti ukuran
kemiripan **Euclidean (EDM-Fuzzy)** dengan **Jensen–Shannon Divergence
(JSD-Fuzzy)** membuat ANN-LM lebih baik mengenali (1) ada tidaknya fault,
(2) jenis/kombinasi fault, dan (3) sensor mana yang rusak?

## 2. Data: 4 sensor, ±98 hari

Sensor mencatat tiap 30 detik; broker menyinkronkan keempatnya lalu merata-rata
per 5 menit. Grafik di bawah: rata-rata per jam, seluruh periode.

In [ ]:
jam = C["ringkas_per_jam"]
hari = np.arange(len(jam)) / 24
fig, ax = plt.subplots(figsize=(11, 3))
for j in range(4):
    ax.plot(hari, jam[:, j], lw=0.8, label=f"S{j+1}")
for h0 in C["hari"]:
    ax.axvspan(h0, h0 + int(C["win"]) * 300 / 86400, color="0.85", zorder=0)
ax.set_xlabel("hari sejak awal rekaman"); ax.set_ylabel("kelembaban (%)")
ax.set_title("Data lapangan (abu-abu = tiga jendela yang dipakai di panel interaktif)")
ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()
print(f"{int(C['n_sampel_5menit']):,} titik 5 menit = {int(C['n_sampel_5menit']) * 300 / 86400:.1f} hari")

In [ ]:
# === Panel kontrol (dipakai bagian 3, 4, dan 6) ===
pil_hari = w.Dropdown(options=[(f"hari {int(h)}", i) for i, h in enumerate(C["hari"])], description="Jendela")
pil_sensor = w.Dropdown(options=[(f"S{s+1}", s) for s in range(4)], value=1, description="Sensor rusak")
pil_fault = w.Dropdown(options=[(f, i) for i, f in enumerate(C["fault"])], value=4, description="Fault")
pil_int = w.Dropdown(options=[(f"×{int(k)}" + (" (penelitian)" if k == 1 else ""), i)
                              for i, k in enumerate(C["intensitas"])], description="Intensitas")
PANEL = w.HBox([pil_hari, pil_sensor, pil_fault, pil_int])
KONTROL = dict(hi=pil_hari, s=pil_sensor, fi=pil_fault, ki=pil_int)

def hidupkan(fn, kontrol=None, panel=None):
    # Tampilkan hasil fn(**pilihan) sebagai output sel biasa (terlihat juga di tampilan
    # statis) lalu perbarui di tempat setiap kali pilihan panel berubah.
    kontrol = kontrol or KONTROL
    display(panel or PANEL)
    pegangan = []
    def render(*_):
        isi = fn(**{k: v.value for k, v in kontrol.items()})
        if not pegangan:
            pegangan.extend(display(x, display_id=True) for x in isi)
        else:
            for h, x in zip(pegangan, isi):
                h.update(x)
        plt.close("all")
    for v in kontrol.values():
        v.observe(render, "value")
    render()

display(PANEL)

## 3. Fault disuntikkan ke data asli

Data lapangan dianggap **bersih**; fault dibuat dengan simulator lalu disuntikkan
ke satu sensor. Empat jenis: **drift** (merayap naik pelan), **spike** (lonjakan
sesaat), **bias** (bergeser tetap), **hardware** (macet/nilai hilang); "semua" =
keempatnya sekaligus.

In [ ]:
_t_jam = np.arange(int(C["win"])) * 300 / 3600
FIG_F, AX_F = plt.subplots(1, 2, figsize=(11, 2.8), gridspec_kw={"width_ratios": [3, 1]})
L_asli, = AX_F[0].plot(_t_jam, _t_jam * 0, color="tab:blue", lw=1, label="asli")
L_rusak, = AX_F[0].plot(_t_jam, _t_jam * 0, color="tab:red", lw=0.9, alpha=0.8, label="diberi fault")
L_beda, = AX_F[1].plot(_t_jam, _t_jam * 0, color="tab:red", lw=0.8)
AX_F[0].set_xlabel("jam dalam jendela (N = 700 = 58 jam)"); AX_F[0].legend(fontsize=8)
AX_F[1].set_title("selisih (fault − asli)", fontsize=9); AX_F[1].set_xlabel("jam")
FIG_F.tight_layout(); plt.close(FIG_F)

def gambar_fault(hi, s, fi, ki):
    # satu figur dipakai ulang; hanya data garis yang diganti (respons cepat)
    bersih, rusak = C["sig_bersih"][hi][:, s], C["sig_fault"][fi, ki, s, hi]
    L_asli.set_ydata(bersih); L_rusak.set_ydata(rusak); L_beda.set_ydata(rusak - bersih)
    AX_F[0].set_ylabel(f"S{s+1} (%)")
    for ax in AX_F:
        ax.relim(); ax.autoscale_view()
    return [FIG_F, Markdown(f"Porsi sampel S{s+1} yang kena fault di jendela ini: "
                            f"**{C['porsi'][fi, ki, s, hi]:.0%}**")]

hidupkan(gambar_fault)

## 4. Entropi multiskala: EDM-Fuzzy vs JSD-Fuzzy

Tiap sensor diringkas menjadi **15 angka** (satu per skala τ = 1..15): seberapa
tidak teratur polanya ketika sinyal dilihat makin kasar. Empat sensor × 15 = **60
fitur** per jendela. Garis putus-putus = setelah fault.

In [ ]:
T_SK = C["fit_fault"].shape[-1]; _tau = np.arange(1, T_SK + 1)
FIG_E, AX_E = plt.subplots(1, 2, figsize=(11, 2.8), sharex=True)
L_E = [(ax.plot(_tau, _tau * 0.0, "o-", ms=3, label="asli")[0],
        ax.plot(_tau, _tau * 0.0, "s--", ms=3, label="fault")[0]) for ax in AX_E]
for ax in AX_E:
    ax.set_xlabel("skala τ"); ax.legend(fontsize=8)
AX_E[0].set_ylabel("entropi"); FIG_E.tight_layout(); plt.close(FIG_E)

def gambar_entropi(hi, s, fi, ki):
    beda = []
    for mi, ax in enumerate(AX_E):
        asli = C["fit_bersih"][mi, hi, s * T_SK:(s + 1) * T_SK]
        rusak = C["fit_fault"][mi, fi, ki, s, hi]
        L_E[mi][0].set_ydata(asli); L_E[mi][1].set_ydata(rusak)
        ax.set_title(f"{C['metode'][mi]} — S{s+1}", fontsize=10)
        ax.relim(); ax.autoscale_view()
        beda.append(np.abs(rusak - asli).mean())
    return [FIG_E, Markdown("Rata-rata perubahan entropi akibat fault: " +
                            " · ".join(f"{C['metode'][mi]} **{beda[mi]:.3f}**" for mi in range(2)))]

hidupkan(gambar_entropi)

## 5. ANN dilatih dengan Levenberg–Marquardt (LM)

60 fitur → jaringan saraf kecil (1–2 lapisan tersembunyi) → kelas. Bobot dicari
dengan LM: langkah **(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr**. λ kecil ≈ Gauss–Newton
(cepat dekat minimum), λ besar ≈ turun gradien (hati-hati jauh dari minimum).
Coba geser λ pada contoh mini di bawah (satu neuron, dua bobot).

In [ ]:
rng_toy = np.random.default_rng(0)
x_toy = np.linspace(0, 4, 30); y_toy = 2.0 * np.tanh(0.8 * x_toy) + rng_toy.normal(0, 0.05, 30)

def lm_mini(lam0, adaptif, iters=25):
    th = np.array([0.2, 3.0]); lam = lam0; hist = []
    for _ in range(iters):
        a, b = th; z = np.tanh(b * x_toy); r = a * z - y_toy
        J = np.column_stack([z, a * x_toy * (1 - z**2)])
        A = J.T @ J; g = J.T @ r
        step = np.linalg.solve(A + lam * np.diag(np.diag(A)) + 1e-12 * np.eye(2), -g)
        th_baru = th + step
        r_baru = th_baru[0] * np.tanh(th_baru[1] * x_toy) - y_toy
        if (r_baru**2).sum() < (r**2).sum():
            th = th_baru; lam = lam / 3 if adaptif else lam
        else:
            lam = lam * 2 if adaptif else lam
        hist.append(((th[0] * np.tanh(th[1] * x_toy) - y_toy) ** 2).mean())
    return th, hist

def gambar_lm(log_lambda, adaptif):
    th, hist = lm_mini(10.0 ** log_lambda, adaptif)
    fig, axs = plt.subplots(1, 2, figsize=(11, 2.6))
    axs[0].semilogy(hist, "o-", ms=3); axs[0].set_xlabel("iterasi"); axs[0].set_ylabel("MSE")
    axs[0].set_title(f"λ awal = {10.0 ** log_lambda:g}, {'adaptif' if adaptif else 'tetap'}", fontsize=10)
    axs[1].plot(x_toy, y_toy, "k.", label="data")
    axs[1].plot(x_toy, th[0] * np.tanh(th[1] * x_toy), "r-", label="hasil fit"); axs[1].legend(fontsize=8)
    plt.tight_layout()
    return [fig]

K_LM = dict(log_lambda=w.FloatSlider(value=-2, min=-4, max=3, step=0.5, description="log10 λ"),
            adaptif=w.Checkbox(value=True, description="λ adaptif (seperti penelitian)"))
hidupkan(gambar_lm, K_LM, w.HBox(list(K_LM.values())))

## 6. Tebakan model tersimpan untuk jendela pilihan

Model N = 700 yang disimpan (`models/`) dipakai langsung: **S1** menjawab "normal
atau rusak", model skenario sesuai jumlah fault menjawab jenisnya, dan model
sensor-ID menunjuk sensor yang rusak (skor > 0,5).

In [ ]:
# Bobot 10 model N = 700 dari models/ (diekspor ke numpy oleh sidang/qa/buat_cache_interaktif.py,
# prediksinya sudah dicek sama dengan model asli; selisih < 1e-9).
MZ = np.load(ROOT / "sidang" / "assets" / "model_sidang.npz")
META = json.loads(str(MZ["_meta"]))

def maju(kunci, X, sensor=False):
    X = np.array(X, float, copy=True)
    nan = np.isnan(X)
    if nan.any():
        X[nan] = np.broadcast_to(MZ[kunci + "|fill"], X.shape)[nan]
    a = (X - MZ[kunci + "|mean"]) / MZ[kunci + "|scale"]
    n, act = int(MZ[kunci + "|n"]), str(MZ[kunci + "|act"])
    for i in range(n):
        a = a @ MZ[f"{kunci}|W{i}"] + MZ[f"{kunci}|b{i}"]
        if i < n - 1:
            a = np.maximum(a, 0.0) if act == "relu" else np.tanh(a)
    if sensor:
        return a
    e = np.exp(a - a.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

MODELS = {}
for k in sorted({"|".join(x.split("|")[:2]) for x in MZ.files if not x.startswith("_")}):
    sc, met = k.split("|")
    MODELS[(sc, met)] = {"class_names": [str(c) for c in MZ[k + "|kelas"]],
                         "detektor": lambda X, k=k: maju(k + "|det", X),
                         "sensor_id": lambda X, k=k: maju(k + "|sen", X, sensor=True),
                         "cv_f1_nb11": float(MZ[k + "|cv_f1_nb11"])}
print(f"{len(MODELS)} model dimuat | dibuat {META['dibuat']} | commit {META['commit']} | N = {META['window']}")

In [ ]:
def tebak(hi, s, fi, ki):
    T = C["fit_fault"].shape[-1]
    sken = "S5" if C["fault"][fi] == "semua" else "S2"
    baris, sens = [], []
    for mi, met in enumerate(C["metode"]):
        f_asli = C["fit_bersih"][mi, hi].copy()
        f_rusak = f_asli.copy(); f_rusak[s * T:(s + 1) * T] = C["fit_fault"][mi, fi, ki, s, hi]
        X2 = np.vstack([f_asli, f_rusak])
        for sc in ["S1", sken]:
            b = MODELS[(sc, met)]; P = b["detektor"](X2)
            for wi, nama in enumerate(["asli", "diberi fault"]):
                k = int(P[wi].argmax())
                baris.append({"Jendela": nama, "Model": sc, "Metode": met,
                              "Tebakan": b["class_names"][k], "Yakin": f"{P[wi, k]:.0%}"})
        skor = MODELS[(sken, met)]["sensor_id"](f_rusak[None])[0]
        sens.append({"Metode": met, **{f"S{j+1}": round(float(v), 2) for j, v in enumerate(skor)},
                     "Ditebak rusak": ", ".join(f"S{j+1}" for j, v in enumerate(skor) if v > 0.5) or "-",
                     "Benar": f"S{s+1}"})
    benar_s = "drift+spike+bias+HW" if sken == "S5" else str(C["fault"][fi])
    return [Markdown(f"**Jawaban benar:** jendela asli = *fault free*; jendela fault = *faulty* (S1) "
                     f"dan *{benar_s}* ({sken}). Intensitas latih = ×1."),
            pd.DataFrame(baris), Markdown("**Sensor mana yang rusak** (skor > 0,5 = rusak):"),
            pd.DataFrame(sens)]

hidupkan(tebak)

## 7. Hasil utama (notebook 11, LM asli, 15 sel = 5 skenario × 3 panjang jendela)

Nilai = F1 makro rata-rata 5 lipatan CV. Hijau = JSD lebih tinggi.

In [ ]:
f1 = pd.read_csv(ROOT / "hasil11" / "exports" / "11_rq5_f1_berpasangan.csv")
f1 = f1[pd.to_numeric(f1["N_window"], errors="coerce").notna()].copy()
f1["N"] = f1["N_window"].astype(float).astype(int)
tabel = f1.pivot(index="Scenario", columns="N", values="Difference")
display(Markdown("**Selisih F1 (JSD − EDM)** per sel:"))
display(tabel.style.format("{:+.3f}").background_gradient(cmap="RdYlGn", vmin=-0.1, vmax=0.1))
r = f1[["EDM-Fuzzy F1", "JSD-Fuzzy F1"]].astype(float)
print(f"JSD menang {int((f1['Difference'].astype(float) > 0).sum())} dari {len(f1)} sel | "
      f"rata-rata F1 EDM {r['EDM-Fuzzy F1'].mean():.3f} vs JSD {r['JSD-Fuzzy F1'].mean():.3f}")
sid = pd.read_csv(ROOT / "hasil11" / "exports" / "11_sensor_ringkas.csv")
display(Markdown("**Identifikasi sensor** (dirantai dari prediksi out-of-fold):")); display(sid)

## 8. Seberapa kuat buktinya?

Uji berpasangan per notebook, ditambah ukuran efek (Cohen dz), interval
kepercayaan bootstrap 95 % untuk selisih rata-rata, dan koreksi Holm untuk tiga
panjang jendela.

In [ ]:
st = pd.read_csv(ROOT / "sidang" / "qa" / "hasil_uji" / "statistik_sel.csv")
kol = ["notebook", "N", "menang_JSD", "n_sel", "rerata_selisih", "p_t", "p_wilcoxon",
       "p_t_holm_3N", "cohen_dz", "ci95_bawah", "ci95_atas"]
display(st[kol].style.format({c: "{:.3f}" for c in kol[4:]}, na_rep="—"))
p_lip = ROOT / "sidang" / "qa" / "hasil_uji" / "statistik_lipatan.csv"
if p_lip.exists():
    display(Markdown("**Tingkat lipatan, N = 700** (arsitektur tetap; t terkoreksi Nadeau–Bengio):"))
    display(pd.read_csv(p_lip).round(3))

## 9. Kelemahan yang kami ketahui (dan apa yang sudah diukur)

In [ ]:
p_ef = ROOT / "sidang" / "qa" / "hasil_uji" / "efek_purge_permutasi.csv"
if p_ef.exists():
    ef = pd.read_csv(p_ef)
    display(Markdown("**Purge** = buang jendela latih yang bertumpang-tindih waktu dengan jendela uji. "
                     "**Permutasi** = label latih diacak; F1 harus turun ke tingkat tebakan."))
    display(ef.round(3))
else:
    print("Hasil uji purge/permutasi belum tersedia.")

| Jenis | Kelemahan | Dampak |
|---|---|---|
| Internal | Jendela bersebelahan berbagi setengah sampel di batas grup CV | skor absolut sedikit optimistis; berlaku sama untuk EDM dan JSD |
| Konstruk | Fault disimulasikan, bukan kerusakan nyata | hasil berlaku untuk bentuk fault yang dimodelkan |
| Konstruk | *Centering* menghapus bias tetap | bias tidak terdeteksi oleh kedua metode |
| Statistik | 15 sel tidak saling bebas; banyak uji | p gabungan dibaca hati-hati, dilengkapi Holm/dz/CI |
| Eksternal | Satu lokasi, 4 sensor, ±98 hari | perlu validasi di lokasi/sensor lain |

## 10. Kesimpulan

1. Entropi multiskala 4 sensor + ANN-LM dapat mendeteksi fault dan menunjuk sensor
   yang rusak, paling baik untuk hardware/macet dan kombinasi banyak fault.
2. Mengganti Euclidean dengan **Jensen–Shannon Divergence** memberi keunggulan yang
   konsisten; kekuatan statistiknya bergantung pada bentuk fault (lihat bagian 8).
3. Bias tetap dan drift kecil tetap sulit: keterbatasan ini berasal dari
   *centering* di dalam entropi, bukan dari classifier.